# 02 — Export to SwarmLabs Config

Take a generated topology and export it as a SwarmLabs-compatible
agent config, ready to import into CrewAI, AutoGen, or LangGraph.

**Prerequisites:**
- Python 3.10+ with numpy
- Running from inside the `flybrain-connectome` checkout
- Completed [01_generate_visualize.ipynb](01_generate_visualize.ipynb)

## Cell 1 — Load a previously generated topology

In [ ]:
import sys, os, json
from pathlib import Path

HERE = Path(__file__).resolve().parents[1]
TOOLKIT = HERE / "flybrain-connectome-mcp"
for p in [TOOLKIT, TOOLKIT / "tools"]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import synthetic_bio_topology as c1

# Load the topology from notebook 01
org_dir = TOOLKIT / "generated_organisms"
json_path = org_dir / "notebook_demo_m5_a8.json"
if not json_path.exists():
    print("[warn] Run 01_generate_visualize.ipynb first")
    # Generate fresh
    topo = c1.design_organism({"n_modules": 5, "agents_per_module": 8})
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(topo, f, indent=2, default=str)
else:
    with open(json_path, "r", encoding="utf-8") as f:
        topo = json.load(f)

print(f"[loaded] {len(topo['agents'])} agents, {len(topo['edges'])} edges")

## Cell 2 — Export to SwarmLabs config

In [ ]:
swarm = c1.build_swarmlabs_config(topo)

sw_dir = TOOLKIT / "swarmlabs"
sw_dir.mkdir(parents=True, exist_ok=True)

name = "notebook_demo_m5_a8"
sw_json = sw_dir / (name + ".swarm.json")
sw_yaml = sw_dir / (name + ".swarm.yaml")

with open(sw_json, "w", encoding="utf-8") as f:
    json.dump(swarm, f, indent=2, default=str)
print("[saved]", sw_json)

# YAML export (if yaml available)
try:
    import yaml
    with open(sw_yaml, "w", encoding="utf-8") as f:
        yaml.dump(swarm, f, default_flow_style=False, allow_unicode=True)
    print("[saved]", sw_yaml)
except ImportError:
    print("[skip] pyyaml not installed; JSON only")

## Cell 3 — Inspect the SwarmLabs config

In [ ]:
print("SwarmLabs config structure:")
print("  agents:", len(swarm.get("agents", [])))
print("  edges :", len(swarm.get("edges", [])))
print()

# Show first 3 agents
for a in swarm.get("agents", [])[:3]:
    print(f"  agent: {a.get('id', '?')}  role={a.get('role', '?')}  model={a.get('model', '?')}")
    print(f"    task: {a.get('task', '')[:80]}...")

## Cell 4 — Import into CrewAI

In [ ]:
# Pseudocode for CrewAI integration (requires crewai installed)
# The SwarmLabs JSON maps directly to CrewAI Agent + Task structures.

crewai_mapping = []
for agent in swarm.get("agents", []):
    crewai_mapping.append({
        "role": agent.get("role", "Agent"),
        "goal": agent.get("task", "Execute task"),
        "backstory": agent.get("backstory", ""),
        "llm": agent.get("model", "gpt-4"),
        "source_neuron": agent.get("provenance", {}).get("source_neuron", "unmapped"),
    })

print(f"Mapped {len(crewai_mapping)} agents for CrewAI import")
for a in crewai_mapping[:3]:
    print(f"  {a['role']:30s}  src={a['source_neuron']:40s}  model={a['llm']}")

## Cell 5 — Import into AutoGen

In [ ]:
# Pseudocode for AutoGen integration (requires autogen installed)
# The topology edges map to AutoGen's message routing.

print("AutoGen team mapping:")
print(f"  Total agents: {len(swarm.get('agents', []))}")
print(f"  Total edges : {len(swarm.get('edges', []))}")
print()
print("Edge routing (first 10):")
for e in swarm.get("edges", [])[:10]:
    src = e.get("source", "?")
    tgt = e.get("target", "?")
    typ = e.get("type", "message")
    print(f"  {src} --[{typ}]--> {tgt}")

## Key insight

The SwarmLabs config isn't just a list of agents — it's a **
provably-grounded agent topology**. Every agent in the config
traces to a specific role in the Drosophila brain, and every edge
traces to a real synapse class.

When your CrewAI or AutoGen agent swarm misbehaves, you can:
1. **Trace** the failing agent back to its neural role
2. **Profile** the topology for stability (spectral radius, clustering)
3. **Audit** the provenance chain to see what biological prior drove
   the design decision

This is the "查不清" (unauditable) problem solved.

## Next

- [03_provenance_audit.ipynb](03_provenance_audit.ipynb) — dive into the provenance chain